# 06. 분석 2 — 유전체 기반 뇌 위축 분석

연구기획서 5절(표 5)을 실행한다. 분석 1에서 유전체는 질환 진단력이 있을 때 추가 가치가 없었고, 진단력이 없을 때만 일부를 메웠다. 이 노트북은 그 결과의 **기전**을 자료로 확인한다: 유전 위험이 뇌 위축에 닿는 경로가 대사질환을 거치는지, 직접 경로도 있는지.

| 절 | 기획서 항목 | 내용 |
| --- | --- | --- |
| 1 | 2-1 | PRS와 APOE의 해마, 전체 뇌, 뇌실 z 연관(연령, 성별, PC 보정), 위축 양성 OR, 십분위, APOE 용량-반응과 상호작용 |
| 2 | 2-2 | lead 178개와 LD 이웃 1,938개의 연관 스캔(Bonferroni), 배경 SNP 997,884개로 λ_GC와 QQ 점검, 형질군별 요약 |
| 3 | 2-4 | 신호 전달 점검: 질환 변수 보정 전후 PRS 효과의 감쇠 |
| 4 | 2-3(a) | 매개분석: PRS → 질환(또는 측정치) → 해마 z, 자연 간접효과와 매개 비율(부트스트랩) |
| 5 | 2-3(b) | 1-표본 멘델리안 무작위화(2SLS): PRS를 도구변수로 대사 형질이 해마 z에 미치는 효과, F 통계량, 내생성 검정, 다중 도구 Sargan 검정, APOE 영역 제외 민감도 |
| 6 | 표 4 | 보고서용 요약표와 그림 6 |

대상은 유전체가 있는 기반군 10,000명 전체다(연관 분석이므로 개발군과 평가군을 나누지 않는다). 다만 PRS 가중치는 노트북 04에서 개발군의 질환 자료로 적합했으므로, PRS와 뇌 z의 연관은 평가군 2,000명만으로도 다시 구해 민감도로 제시한다. 유전체 자료는 로컬에서만 읽고 외부로 보내지 않는다.

`FAST = True`이면 배경 스캔을 21, 22번 염색체로, 부트스트랩을 200회로 줄인다(코드 점검용).

In [8]:
FAST = False

import json, time, warnings, re
from pathlib import Path
import numpy as np, pandas as pd
from scipy import stats
from scipy.special import expit
import statsmodels.api as sm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import logging; logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
import matplotlib.font_manager as fm
warnings.filterwarnings("ignore")
_avail = {f.name for f in fm.fontManager.ttflist}
plt.rcParams["font.family"] = [f for f in ["AppleGothic", "Malgun Gothic", "NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"] if f in _avail] + ["DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
COL = {"blue": "#2a78d6", "orange": "#eb6834", "aqua": "#1baf7a", "violet": "#4a3aa7", "gray": "#9aa3ad", "ink": "#0b0b0b", "ink2": "#52514e"}

ROOT = Path.cwd()
while not (ROOT / "02_Data").exists() and ROOT != ROOT.parent: ROOT = ROOT.parent
GDIR = ROOT / "02_Data" / "유전체_합성데이터"
OUT = ROOT / "05_분석결과"; DERIVED, TAB, FIG = OUT / "derived", OUT / "tables", OUT / "figures"
SEED = 20260915; NBOOT = 200 if FAST else 1000; PFX = "fast_" if FAST else ""
rng = np.random.default_rng(SEED)

B = pd.read_csv(DERIVED / "base_analysis.csv"); Gf = pd.read_csv(DERIVED / "genomic_features.csv")
A = B.merge(Gf, on="T_ID", how="inner").reset_index(drop=True)
ann = pd.read_csv(DERIVED / "lead_annotation.csv"); spec = json.load(open(DERIVED / "genomic_features_spec.json", encoding="utf-8"))
LEAD = pd.read_csv(DERIVED / "lead_dosage.csv").set_index("T_ID").loc[A.T_ID]           # 10,000 × 178, A1(minor) 개수
PCS = [f"pc{i}" for i in range(1, 11)]; COV = ["age", "male"] + PCS
PRS = ["prs_t2d", "prs_htn", "prs_lipid", "prs_obesity", "prs_alcohol"]
for p in PRS: A[p + "_s"] = (A[p] - A[p].mean()) / A[p].std()                         # 1 SD 단위
A["e4"], A["e2"] = A.apoe_e4_count, A.apoe_e2_count
FACTORS = {"prs_t2d_s": "T2D PRS", "prs_htn_s": "HTN PRS", "prs_lipid_s": "LIPID PRS", "prs_obesity_s": "OBESITY PRS", "prs_alcohol_s": "ALCOHOL PRS", "e4": "APOE ε4 개수", "e2": "APOE ε2 개수"}
OUTS = {"z_hip": "해마 z", "z_wb": "전체 뇌 z", "z_vent": "뇌실 z"}
DZ = ["htn", "dm", "lip"]; MEAS = {"htn": "sbp", "dm": "glu", "lip": "ldl"}; PRS_OF = {"htn": "prs_htn_s", "dm": "prs_t2d_s", "lip": "prs_lipid_s"}
ev = (A.split == "eval").values
print(f"대상자 {len(A)} (평가군 {int(ev.sum())}) | lead 변이 {LEAD.shape[1]} | PRS 형질군 집합 크기 {dict((k, len(v)) for k, v in spec['prs_sets'].items())}")
print("질환 유병률:", A[DZ].mean().round(3).to_dict(), "| 결측:", A[DZ + list(MEAS.values())].isna().sum().to_dict())

def ols(y, Xcols, d=None, robust=False):
    d = A if d is None else d
    dd = d.dropna(subset=[y] + Xcols)
    m = sm.OLS(dd[y].values, sm.add_constant(dd[Xcols].values)).fit(cov_type="HC1" if robust else "nonrobust")
    m.names = ["const"] + Xcols; return m
def coef(m, name):
    i = m.names.index(name); ci = m.conf_int()[i]
    return float(m.params[i]), float(ci[0]), float(ci[1]), float(m.pvalues[i])
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty_like(p); run = 0.0
    for r, i in enumerate(o): run = max(run, min(1.0, (len(p) - r) * p[i])); adj[i] = run
    return adj
def fmt(b, lo, hi, nd=3): return f"{b:+.{nd}f} ({lo:+.{nd}f}, {hi:+.{nd}f})"

대상자 10000 (평가군 2000) | lead 변이 178 | PRS 형질군 집합 크기 {'T2D': 36, 'HTN': 34, 'LIPID': 32, 'OBESITY': 5}
질환 유병률: {'htn': 0.201, 'dm': 0.071, 'lip': 0.079} | 결측: {'htn': 18, 'dm': 19, 'lip': 22, 'sbp': 41, 'glu': 236, 'ldl': 196}


## 1. PRS와 APOE의 뇌 체적 연관 (2-1)

In [9]:
rows = []
for f, flab in FACTORS.items():
    for o, olab in OUTS.items():
        b, lo, hi, p = coef(ols(o, [f] + COV), f)
        be, loe, hie, pe = coef(ols(o, [f] + COV, A[ev]), f)
        r = {"유전 요인": flab, "결과": olab, "β(1 SD 또는 대립유전자 1개당)": b, "CI 하한": lo, "CI 상한": hi, "p": p, "평가군만 β": be, "평가군만 p": pe}
        if o == "z_hip":   # 위축 양성 OR
            dd = A.dropna(subset=[f] + COV); lg = sm.Logit(dd.atrophy_pos.values, sm.add_constant(dd[[f] + COV].values)).fit(disp=0)
            r["위축 양성 OR"] = float(np.exp(lg.params[1])); r["OR 하한"], r["OR 상한"] = np.exp(lg.conf_int()[1]); r["OR p"] = float(lg.pvalues[1])
        rows.append(r)
T4a = pd.DataFrame(rows); T4a["Holm p (21검정)"] = holm(T4a.p)
T4a.to_csv(TAB / f"{PFX}table4a_prs_apoe_association.csv", index=False, encoding="utf-8-sig")
show = T4a[T4a.결과 == "해마 z"].copy()
show["β (95% CI)"] = [fmt(b, lo, hi) for b, lo, hi in zip(show["β(1 SD 또는 대립유전자 1개당)"], show["CI 하한"], show["CI 상한"])]
show["OR (95% CI)"] = [f"{o:.3f} ({l:.3f}, {h:.3f})" for o, l, h in zip(show["위축 양성 OR"], show["OR 하한"], show["OR 상한"])]
print("해마 z에 대한 효과(연령, 성별, PC1–10 보정; 전체 10,000명)")
print(show[["유전 요인", "β (95% CI)", "p", "Holm p (21검정)", "OR (95% CI)", "평가군만 β", "평가군만 p"]].to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("\n전체 뇌 z와 뇌실 z에서 Holm 유의(0.05) 요인:",
      T4a[(T4a.결과 != "해마 z") & (T4a["Holm p (21검정)"] < 0.05)][["유전 요인", "결과", "β(1 SD 또는 대립유전자 1개당)"]].round(3).values.tolist() or "없음")

# 십분위: 상위 십분위 대 하위 십분위
dec_rows = []
for f, flab in FACTORS.items():
    if f.startswith("e") or f == "prs_alcohol_s": continue          # 알코올 PRS는 변이 2개라 십분위가 정의되지 않음
    q = pd.qcut(A[f].rank(method="first"), 10, labels=False); top, bot = A[q == 9], A[q == 0]
    dec_rows.append({"PRS": flab, "하위 10% 해마 z": bot.z_hip.mean(), "상위 10% 해마 z": top.z_hip.mean(), "차이": top.z_hip.mean() - bot.z_hip.mean(),
                     "하위 10% 위축률": bot.atrophy_pos.mean(), "상위 10% 위축률": top.atrophy_pos.mean(),
                     "상위 10% 질환 유병률(해당 질환)": {"prs_t2d_s": top.dm.mean(), "prs_htn_s": top.htn.mean(), "prs_lipid_s": top.lip.mean()}.get(f, np.nan),
                     "하위 10% 질환 유병률(해당 질환)": {"prs_t2d_s": bot.dm.mean(), "prs_htn_s": bot.htn.mean(), "prs_lipid_s": bot.lip.mean()}.get(f, np.nan)})
DEC = pd.DataFrame(dec_rows); DEC.to_csv(TAB / f"{PFX}table4a2_prs_deciles.csv", index=False, encoding="utf-8-sig")
print("\nPRS 십분위 비교(상위 10% 대 하위 10%):"); print(DEC.round(3).to_string(index=False))

# APOE 용량-반응과 상호작용
print("\nAPOE ε4 개수별 해마 z 평균(연령, 성별 미보정):", A.groupby("e4").z_hip.agg(["count", "mean"]).round(3).to_dict("index"))
A["age_c"] = A.age - A.age.mean(); A["e4_x_age"] = A.e4 * A.age_c; A["e4_x_dz"] = A.e4 * A.any_dz
m_dose = ols("z_hip", ["e4"] + COV); m_int_age = ols("z_hip", ["e4", "e4_x_age"] + COV); m_int_dz = ols("z_hip", ["e4", "any_dz", "e4_x_dz"] + COV)
A["e4_1"], A["e4_2"] = (A.e4 == 1).astype(int), (A.e4 == 2).astype(int); m_cat = ols("z_hip", ["e4_1", "e4_2"] + COV)
apoe = {"ε4 1개당 β": fmt(*coef(m_dose, "e4")[:3]), "ε4 1개(대 0개) β": fmt(*coef(m_cat, "e4_1")[:3]), "ε4 2개(대 0개) β": fmt(*coef(m_cat, "e4_2")[:3]),
        "ε4×연령(10세당) β": fmt(*(np.array(coef(m_int_age, "e4_x_age")[:3]) * 10)), "ε4×연령 p": coef(m_int_age, "e4_x_age")[3],
        "ε4×대사질환 β": fmt(*coef(m_int_dz, "e4_x_dz")[:3]), "ε4×대사질환 p": coef(m_int_dz, "e4_x_dz")[3],
        "ε4와 고지혈증 진단 OR": float(np.exp(sm.Logit(A.dropna(subset=["lip"]).lip.values, sm.add_constant(A.dropna(subset=["lip"])[["e4"] + COV].values)).fit(disp=0).params[1]))}
pd.Series(apoe).to_csv(TAB / f"{PFX}table4a3_apoe.csv", header=["값"], encoding="utf-8-sig")
print("APOE:", json.dumps({k: (round(v, 4) if isinstance(v, float) else v) for k, v in apoe.items()}, ensure_ascii=False, indent=1))

해마 z에 대한 효과(연령, 성별, PC1–10 보정; 전체 10,000명)
      유전 요인              β (95% CI)        p  Holm p (21검정)          OR (95% CI)  평가군만 β   평가군만 p
    T2D PRS -0.026 (-0.046, -0.007)  0.00894          0.143 1.059 (1.003, 1.118) -0.0386   0.0807
    HTN PRS -0.095 (-0.115, -0.076) 2.58e-21       5.42e-20 1.253 (1.186, 1.323)  -0.089 5.36e-05
  LIPID PRS -0.019 (-0.038, +0.001)   0.0637          0.801 1.008 (0.954, 1.065) -0.0023    0.914
OBESITY PRS -0.033 (-0.052, -0.013)  0.00101         0.0172 1.104 (1.046, 1.165) -0.0188    0.399
ALCOHOL PRS -0.019 (-0.039, +0.000)   0.0547          0.766 1.049 (0.994, 1.107) -0.0117    0.592
 APOE ε4 개수 +0.006 (-0.042, +0.055)    0.794              1 1.001 (0.875, 1.145)  0.0808    0.137
 APOE ε2 개수 +0.006 (-0.043, +0.054)    0.816              1 0.923 (0.803, 1.061) 0.00992    0.858

전체 뇌 z와 뇌실 z에서 Holm 유의(0.05) 요인: [['T2D PRS', '전체 뇌 z', -0.033], ['HTN PRS', '전체 뇌 z', -0.062], ['HTN PRS', '뇌실 z', 0.065]]

PRS 십분위 비교(상위 10% 대 하위 10%):
        PRS  하위 10

## 2. 코호트 내 연관 스캔 (2-2)

lead 178개와 LD 이웃 1,938개(총 2,116개)에 대해 연령, 성별, PC1–10을 보정한 선형 회귀로 해마, 전체 뇌, 뇌실 z와 질환 및 측정치 연관을 구한다(Frisch–Waugh–Lovell: 결과와 유전형을 공변량으로 잔차화한 뒤 회귀, 단일 공변량 모형과 동일한 계수와 표준오차). 배경 SNP 997,884개는 설계상 null이므로 해마 z 스캔의 λ_GC와 QQ로 검정 보정을 확인한다. 유의 기준은 변이 2,116개에 대한 Bonferroni(0.05/2,116)다.

In [10]:
fam = pd.read_csv(GDIR / "synthetic_kchip_chr1.fam", sep=r"\s+", header=None, names=["FID", "IID", "PAT", "MAT", "SEX", "PHENO"])
N = len(fam); BPS = (N + 3) // 4
POS = pd.Index(fam.IID.astype(str)).get_indexer(A.T_ID.astype(str)); assert (POS >= 0).all()
LUT = np.array([2, np.nan, 1, 0], dtype=np.float32)
bims = {c: pd.read_csv(GDIR / f"synthetic_kchip_chr{c}.bim", sep=r"\s+", header=None, names=["CHR", "SNP", "CM", "BP", "A1", "A2"]) for c in range(1, 23)}

def read_block(chrom, start, count):
    """연속 count개 SNP를 읽어 (count × n, A 순서) 소수 대립유전자 개수 행렬로 돌려준다."""
    with open(GDIR / f"synthetic_kchip_chr{chrom}.bed", "rb") as f:
        f.seek(3 + start * BPS); buf = np.frombuffer(f.read(BPS * count), dtype=np.uint8).reshape(-1, BPS)
    bits = np.unpackbits(buf, axis=1, bitorder="little")[:, :2 * N].reshape(buf.shape[0], N, 2)
    return LUT[bits[:, :, 0] + 2 * bits[:, :, 1]][:, POS]

class FWL:
    """공변량으로 잔차화한 뒤 변이별 단순회귀(완전 사례)."""
    def __init__(self, ycol, covs=COV):
        self.mask = A[[ycol] + covs].notna().all(axis=1).values
        X = np.column_stack([np.ones(self.mask.sum()), A.loc[self.mask, covs].values]).astype(np.float64)
        self.X = X; self.H = X @ np.linalg.inv(X.T @ X); self.p = X.shape[1]; self.n = X.shape[0]
        y = A.loc[self.mask, ycol].values.astype(np.float64); self.yr = y - self.H @ (X.T @ y); self.yy = float(self.yr @ self.yr)
    def scan(self, Gkn):          # Gkn: k × n(전체 A 순서) → beta, se, p (k)
        G = Gkn[:, self.mask].T.astype(np.float64)                      # n × k
        G = np.where(np.isnan(G), np.nanmean(G, axis=0, keepdims=True), G)
        Gr = G - self.H @ (self.X.T @ G); gg = (Gr ** 2).sum(0)
        ok = gg > 1e-8; beta = np.full(G.shape[1], np.nan); se = beta.copy()
        beta[ok] = (Gr[:, ok].T @ self.yr) / gg[ok]
        rss = np.maximum(self.yy - beta[ok] ** 2 * gg[ok], 1e-12); df = self.n - self.p - 1
        se[ok] = np.sqrt(rss / df / gg[ok]); t = beta / se
        return beta, se, 2 * stats.t.sf(np.abs(t), df)

# (1) lead + LD 이웃
t0 = time.time(); mats, names, chrs = [], [], []
for c in range(1, 23):
    bim = bims[c]; idx = bim.index[bim.SNP.str.match(r"^rs\d+(_ld\d+)?$")].to_numpy()
    if len(idx) == 0: continue
    rows_ = np.vstack([read_block(c, int(i), 1) for i in idx]); mats.append(rows_); names += bim.SNP[idx].tolist(); chrs += [c] * len(idx)
GL = np.vstack(mats); del mats
V = pd.DataFrame({"snp": names, "chr": chrs}); V["locus"] = V.snp.str.replace(r"_ld\d+$", "", regex=True); V["kind"] = np.where(V.snp.str.contains("_ld"), "LD 이웃", "lead")
V["maf"] = np.nanmean(GL, axis=1) / 2
grp = ann.set_index("snp").group; V["group"] = V.locus.map(grp).fillna("OTHER")
PHEN = ["z_hip", "z_wb", "z_vent", "htn", "dm", "lip", "sbp", "glu", "ldl", "bmi"]
for ph in PHEN:
    b, s, p = FWL(ph).scan(GL); V[f"beta_{ph}"], V[f"se_{ph}"], V[f"p_{ph}"] = b, s, p
BONF = 0.05 / len(V); V.to_csv(DERIVED / f"{PFX}scan_lead_ld.csv", index=False)
print(f"lead+LD {len(V)}개 × 표현형 {len(PHEN)}개 스캔 완료 ({time.time() - t0:.0f}초) | Bonferroni p < {BONF:.2e}")

# (2) 배경 SNP: 해마, 전체 뇌, 뇌실 z
t0 = time.time(); F3 = {o: FWL(o) for o in OUTS}; bgp = {o: [] for o in OUTS}; n_bg = 0; CHROMS = [21, 22] if FAST else range(1, 23); BLK = 4000
for c in CHROMS:
    bim = bims[c]; isbg = bim.SNP.str.startswith("bg_").to_numpy()
    for s in range(0, len(bim), BLK):
        blk = read_block(c, s, min(BLK, len(bim) - s)); keep = isbg[s:s + blk.shape[0]]
        if keep.sum() == 0: continue
        blk = blk[keep]; n_bg += blk.shape[0]
        for o in OUTS: bgp[o].append(F3[o].scan(blk)[2].astype(np.float32))
bgp = {o: np.concatenate(v) for o, v in bgp.items()}
for o in OUTS: np.save(DERIVED / f"{PFX}bg_scan_p_{o}.npy", bgp[o])
lam = {o: float(np.nanmedian(stats.chi2.isf(bgp[o], 1)) / 0.4549) for o in OUTS}
print(f"배경 SNP {n_bg:,}개 스캔 완료 ({time.time() - t0:.0f}초) | λ_GC:", {OUTS[o]: round(v, 3) for o, v in lam.items()},
      "| 배경에서 p < 5e-8:", {OUTS[o]: int((bgp[o] < 5e-8).sum()) for o in OUTS}, "| 기대 거짓양성(p<Bonferroni 2,116):", round(n_bg * BONF, 1), "관측:", {OUTS[o]: int((bgp[o] < BONF).sum()) for o in OUTS})

# (3) 형질군별 요약(변이 단위와 좌위 단위)
def summarize(df):
    out = {"변이 수": len(df), "좌위 수": df.locus.nunique()}
    for ph, lab in [("z_hip", "해마"), ("z_wb", "전체 뇌"), ("z_vent", "뇌실"), ("htn", "고혈압"), ("dm", "당뇨"), ("lip", "고지혈증"), ("sbp", "SBP"), ("glu", "혈당"), ("ldl", "LDL")]:
        sig = df[f"p_{ph}"] < BONF
        out[f"{lab} 유의 변이"] = int(sig.sum()); out[f"{lab} 유의 좌위"] = df.loc[sig, "locus"].nunique(); out[f"{lab} 최소 p"] = float(df[f"p_{ph}"].min())
    return out
T4b = pd.DataFrame([{"형질군": g, **summarize(V[V.group == g])} for g in ["T2D", "HTN", "LIPID", "OBESITY", "ALCOHOL", "APOE", "OTHER"]] + [{"형질군": "전체", **summarize(V)}])
T4b.to_csv(TAB / f"{PFX}table4b_scan_by_group.csv", index=False, encoding="utf-8-sig")
print("\n형질군별 Bonferroni 유의 변이 수(좌위 수):")
print(T4b[["형질군", "변이 수", "좌위 수"] + [c for c in T4b.columns if c.endswith("유의 변이")]].to_string(index=False))
hit = V[V.p_z_hip < BONF].sort_values("p_z_hip")
print(f"\n해마 z 유의 변이 {len(hit)}개 (좌위 {hit.locus.nunique()}개):")
print(hit[["snp", "chr", "group", "maf", "beta_z_hip", "p_z_hip", "p_htn", "p_dm", "p_lip", "p_sbp"]].head(12).to_string(index=False, float_format=lambda v: f"{v:.3g}"))

# QQ 그림
fig, ax = plt.subplots(figsize=(5.2, 5))
def qq(pv, color, label, maxpts=20000):
    pv = np.sort(pv[~np.isnan(pv)]); n = len(pv); exp = -np.log10((np.arange(1, n + 1) - 0.5) / n); obs = -np.log10(np.maximum(pv, 1e-300))
    if n > maxpts: sel = np.unique(np.concatenate([np.arange(0, 2000), np.linspace(2000, n - 1, maxpts - 2000).astype(int)])); exp, obs = exp[sel], obs[sel]
    ax.scatter(exp, obs, s=6, color=color, label=label, alpha=0.8, linewidths=0)
METAB = ["HTN", "T2D", "LIPID", "OBESITY"]
qq(bgp["z_hip"], COL["gray"], f"배경 SNP {n_bg:,}개 (λ = {lam['z_hip']:.3f})")
qq(V.loc[V.group.isin(METAB), "p_z_hip"].values, COL["blue"], f"대사질환 형질군 lead + LD {int(V.group.isin(METAB).sum()):,}개")
qq(V.loc[~V.group.isin(METAB), "p_z_hip"].values, COL["orange"], f"그 외 형질군(APOE, 알코올, 기타) {int((~V.group.isin(METAB)).sum()):,}개")
m = max(ax.get_xlim()[1], 1); ax.plot([0, m], [0, m], color=COL["ink2"], lw=1, ls="--"); ax.axhline(-np.log10(BONF), color=COL["violet"], lw=1, ls=":")
ax.text(m * 0.98, -np.log10(BONF) + 0.08, "Bonferroni(2,116)", color=COL["violet"], fontsize=8, ha="right")
ax.set_xlabel("기대 −log10 p"); ax.set_ylabel("관측 −log10 p"); ax.set_title("해마 z 연관 스캔 QQ"); ax.legend(fontsize=8, loc="lower right", frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig(FIG / f"{PFX}fig_scan_qq.png", dpi=150); plt.close(); print("그림 저장:", f"{PFX}fig_scan_qq.png")

lead+LD 2116개 × 표현형 10개 스캔 완료 (2초) | Bonferroni p < 2.36e-05
배경 SNP 997,884개 스캔 완료 (344초) | λ_GC: {'해마 z': 1.001, '전체 뇌 z': 1.002, '뇌실 z': 1.008} | 배경에서 p < 5e-8: {'해마 z': 0, '전체 뇌 z': 0, '뇌실 z': 0} | 기대 거짓양성(p<Bonferroni 2,116): 23.6 관측: {'해마 z': 24, '전체 뇌 z': 27, '뇌실 z': 23}

형질군별 Bonferroni 유의 변이 수(좌위 수):
    형질군  변이 수  좌위 수  해마 유의 변이  전체 뇌 유의 변이  뇌실 유의 변이  고혈압 유의 변이  당뇨 유의 변이  고지혈증 유의 변이  SBP 유의 변이  혈당 유의 변이  LDL 유의 변이
    T2D   680    34         0           0         0          0       390           0          0        55          0
    HTN   582    31         3           0         0        340         0           0         18         0          0
  LIPID   521    27         0           0         0          0        21         359          0         0          0
OBESITY    22     3         0           0         0          0        20           0          0         1          0
ALCOHOL    21     2         0           0         0         20         0           0          0         0

## 3. 신호 전달 구조 점검 (2-4)

유전 효과가 질환을 거쳐서만 작용하면, 질환 변수를 보정했을 때 PRS와 해마 z의 연관이 사라져야 한다. 보정 단계를 늘려 가며 PRS 계수의 감쇠를 본다: (a) 연령, 성별, PC → (b) + 질환 진단(고혈압, 당뇨, 고지혈증) → (c) + 측정치(SBP, 혈당, LDL, BMI).

In [11]:
STEPS = {"(a) 연령, 성별, PC": [], "(b) + 질환 진단": DZ, "(c) + 질환 진단 + 측정치": DZ + ["sbp", "glu", "ldl", "bmi"]}
def attenuation(f, steps=STEPS, B=NBOOT):
    base = ols("z_hip", [f] + COV + steps["(b) + 질환 진단"] + [c for c in ["sbp", "glu", "ldl", "bmi"]])  # 완전 사례 공통 표본
    d = A.dropna(subset=[f, "z_hip"] + COV + DZ + ["sbp", "glu", "ldl", "bmi"]).reset_index(drop=True)
    est = {k: coef(ols("z_hip", [f] + COV + v, d), f)[0] for k, v in steps.items()}
    boots = {k: [] for k in steps}
    for _ in range(B):
        i = rng.integers(0, len(d), len(d)); db = d.iloc[i]
        for k, v in steps.items(): boots[k].append(coef(ols("z_hip", [f] + COV + v, db), f)[0])
    rows = []
    for k in steps:
        bb = np.array(boots[k]); att = 1 - bb / np.array(boots["(a) 연령, 성별, PC"])
        rows.append({"PRS": FACTORS[f], "보정": k, "β(1 SD당)": est[k], "CI 하한": np.percentile(bb, 2.5), "CI 상한": np.percentile(bb, 97.5),
                     "감쇠율": 1 - est[k] / est["(a) 연령, 성별, PC"], "감쇠율 CI 하한": np.percentile(att, 2.5), "감쇠율 CI 상한": np.percentile(att, 97.5), "n": len(d)})
    return rows
t0 = time.time(); T4c = pd.DataFrame(sum([attenuation(f) for f in ["prs_htn_s", "prs_t2d_s", "prs_lipid_s", "prs_obesity_s"]], []))
T4c.to_csv(TAB / f"{PFX}table4c_signal_transfer.csv", index=False, encoding="utf-8-sig")
T4c["β (95% CI)"] = [fmt(b, lo, hi, 4) for b, lo, hi in zip(T4c["β(1 SD당)"], T4c["CI 하한"], T4c["CI 상한"])]
T4c["감쇠율 (95% CI)"] = [f"{a:.0%} ({lo:.0%}, {hi:.0%})" if k != "(a) 연령, 성별, PC" else "" for a, lo, hi, k in zip(T4c.감쇠율, T4c["감쇠율 CI 하한"], T4c["감쇠율 CI 상한"], T4c.보정)]
print(f"PRS → 해마 z 계수의 보정 단계별 감쇠 (부트스트랩 {NBOOT}회, {time.time() - t0:.0f}초)")
print(T4c[["PRS", "보정", "β (95% CI)", "감쇠율 (95% CI)"]].to_string(index=False))

# 해마 z 유의 변이(있을 때)에서도 같은 점검
if len(hit):
    rows = []
    for snp in hit.locus.drop_duplicates().head(5):
        if snp not in LEAD.columns: continue
        A["_g"] = LEAD[snp].values
        rows.append({"변이": snp, "형질군": grp.get(snp, "OTHER"), "(a) β": coef(ols("z_hip", ["_g"] + COV), "_g")[0], "(b) +질환 β": coef(ols("z_hip", ["_g"] + COV + DZ), "_g")[0],
                     "(b) p": coef(ols("z_hip", ["_g"] + COV + DZ), "_g")[3]})
    print("\n해마 z 유의 lead 변이의 질환 보정 전후:"); print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.3g}"))

PRS → 해마 z 계수의 보정 단계별 감쇠 (부트스트랩 1000회, 43초)
        PRS                보정                 β (95% CI)      감쇠율 (95% CI)
    HTN PRS    (a) 연령, 성별, PC -0.0954 (-0.1143, -0.0756)                  
    HTN PRS       (b) + 질환 진단 -0.0047 (-0.0258, +0.0180)   95% (77%, 124%)
    HTN PRS (c) + 질환 진단 + 측정치 -0.0046 (-0.0252, +0.0181)   95% (77%, 124%)
    T2D PRS    (a) 연령, 성별, PC -0.0278 (-0.0467, -0.0079)                  
    T2D PRS       (b) + 질환 진단 +0.0144 (-0.0056, +0.0372)  152% (87%, 534%)
    T2D PRS (c) + 질환 진단 + 측정치 +0.0146 (-0.0052, +0.0375)  153% (87%, 538%)
  LIPID PRS    (a) 연령, 성별, PC -0.0165 (-0.0380, +0.0037)                  
  LIPID PRS       (b) + 질환 진단 -0.0016 (-0.0236, +0.0189) 90% (-683%, 804%)
  LIPID PRS (c) + 질환 진단 + 측정치 -0.0015 (-0.0232, +0.0191) 91% (-708%, 815%)
OBESITY PRS    (a) 연령, 성별, PC -0.0335 (-0.0531, -0.0152)                  
OBESITY PRS       (b) + 질환 진단 -0.0019 (-0.0211, +0.0166)   94% (58%, 208%)
OBESITY PRS (c) + 질환 진단 + 측정치 -0.0015 (-0.0205, +0.0170)

## 4. 매개분석 (2-3a)

노출 = PRS(평균에서 1 SD 증가), 매개변수 = 해당 질환 진단(이분형) 또는 측정치(연속), 결과 = 해마 z. 회귀 기반 자연효과 분해(VanderWeele)로 자연 간접효과(NIE), 자연 직접효과(NDE), 총효과(NIE + NDE), 매개 비율을 구하고 부트스트랩으로 신뢰구간을 얻는다. 매개변수 모형이 로지스틱이면 이 총효과는 선형 회귀의 주변 기울기와 조금 다를 수 있어 후자를 참고로 함께 저장한다. 노출과 매개변수의 상호작용은 별도로 검정해 유의하면 상호작용 포함 모형으로 보고한다. 공변량은 연령, 성별, PC1–10이다.

In [12]:
def _logit_fit(X, y, iters=25):
    """IRLS 로지스틱(빠른 부트스트랩용)."""
    b = np.zeros(X.shape[1])
    for _ in range(iters):
        mu = expit(X @ b); W = mu * (1 - mu) + 1e-9
        step = np.linalg.solve((X * W[:, None]).T @ X + 1e-8 * np.eye(X.shape[1]), X.T @ (y - mu)); b += step
        if np.abs(step).max() < 1e-8: break
    return b
def _ols_fit(X, y): return np.linalg.lstsq(X, y, rcond=None)[0]

def natural_effects(d, a, m, y, covs, binary_m, interaction=False):
    C = d[covs].values; one = np.ones(len(d)); av, mv, yv = d[a].values, d[m].values, d[y].values
    Xm = np.column_stack([one, av, C]); bm = _logit_fit(Xm, mv) if binary_m else _ols_fit(Xm, mv)
    cols = [one, av, mv] + ([av * mv] if interaction else []) + [C]; Xy = np.column_stack(cols); by = _ols_fit(Xy, yv)
    th_a, th_m = by[1], by[2]; th_am = by[3] if interaction else 0.0
    a0, a1 = 0.0, 1.0                                                     # PRS 평균에서 +1 SD
    if binary_m:
        lin0 = bm[0] + bm[1] * a0 + C @ bm[2:]; lin1 = bm[0] + bm[1] * a1 + C @ bm[2:]
        Em0, Em1 = expit(lin0), expit(lin1)
    else:
        Em0 = bm[0] + bm[1] * a0 + C @ bm[2:]; Em1 = Em0 + bm[1] * (a1 - a0)
    nde = (th_a * (a1 - a0) + th_am * Em0 * (a1 - a0)).mean()
    nie = ((th_m + th_am * a1) * (Em1 - Em0)).mean()
    lin_tot = _ols_fit(np.column_stack([one, av, C]), yv)[1]            # 선형 주변 기울기(참고)
    return nie, nde, nie + nde, (nie / (nie + nde) if (nie + nde) != 0 else np.nan), lin_tot

def mediation(a, m, y="z_hip", covs=COV, B=NBOOT):
    d = A.dropna(subset=[a, m, y] + covs).reset_index(drop=True); binary_m = set(np.unique(d[m])) <= {0, 1}
    # 상호작용 검정
    d["_am"] = d[a] * d[m]; p_int = coef(ols(y, [a, m, "_am"] + covs, d), "_am")[3]; inter = p_int < 0.05
    est = natural_effects(d, a, m, y, covs, binary_m, inter)
    bs = np.array([natural_effects(d.iloc[rng.integers(0, len(d), len(d))], a, m, y, covs, binary_m, inter) for _ in range(B)])
    lo, hi = np.nanpercentile(bs, 2.5, axis=0), np.nanpercentile(bs, 97.5, axis=0)
    return {"노출": FACTORS[a], "매개변수": m, "n": len(d), "상호작용 p": p_int, "상호작용 포함": inter,
            "NIE": est[0], "NIE 하한": lo[0], "NIE 상한": hi[0], "NDE": est[1], "NDE 하한": lo[1], "NDE 상한": hi[1],
            "총효과": est[2], "총효과 하한": lo[2], "총효과 상한": hi[2], "매개 비율": est[3], "매개 비율 하한": lo[3], "매개 비율 상한": hi[3],
            "선형 총효과(참고)": est[4], "NIE와 NDE 부호 불일치": bool(np.sign(est[0]) != np.sign(est[1]))}

t0 = time.time(); rows = []
for dz in DZ:
    rows.append(mediation(PRS_OF[dz], dz)); rows.append(mediation(PRS_OF[dz], MEAS[dz]))
rows.append(mediation("prs_obesity_s", "bmi")); rows.append(mediation("prs_htn_s", "dm")); rows.append(mediation("prs_t2d_s", "htn"))   # 교차 경로(음성 대조)
T4d = pd.DataFrame(rows); T4d.to_csv(TAB / f"{PFX}table4d_mediation.csv", index=False, encoding="utf-8-sig")
T4d["NIE (95% CI)"] = [fmt(a, b, c, 4) for a, b, c in zip(T4d.NIE, T4d["NIE 하한"], T4d["NIE 상한"])]
T4d["NDE (95% CI)"] = [fmt(a, b, c, 4) for a, b, c in zip(T4d.NDE, T4d["NDE 하한"], T4d["NDE 상한"])]
T4d["총효과 (95% CI)"] = [fmt(a, b, c, 4) for a, b, c in zip(T4d.총효과, T4d["총효과 하한"], T4d["총효과 상한"])]
T4d["매개 비율 (95% CI)"] = [f"{a:.0%} ({b:.0%}, {c:.0%})" for a, b, c in zip(T4d["매개 비율"], T4d["매개 비율 하한"], T4d["매개 비율 상한"])]
print(f"매개분석: PRS 1 SD → 매개변수 → 해마 z (부트스트랩 {NBOOT}회, {time.time() - t0:.0f}초)")
print(T4d[["노출", "매개변수", "총효과 (95% CI)", "NIE (95% CI)", "NDE (95% CI)", "매개 비율 (95% CI)", "상호작용 p", "NIE와 NDE 부호 불일치"]].to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("총효과 = NIE + NDE(자연효과 정의). 부호가 불일치하면 매개 비율이 100%를 넘을 수 있으므로 NIE와 NDE를 각각 해석한다.")

# 다중 매개(세 질환 동시): 차이법 (총효과 − 직접효과)
def multi_med(a, meds, B=NBOOT):
    d = A.dropna(subset=[a, "z_hip"] + COV + meds).reset_index(drop=True)
    def f(db): tot = coef(ols("z_hip", [a] + COV, db), a)[0]; dire = coef(ols("z_hip", [a] + COV + meds, db), a)[0]; return tot, dire, tot - dire, (tot - dire) / tot if tot else np.nan
    e = f(d); bs = np.array([f(d.iloc[rng.integers(0, len(d), len(d))]) for _ in range(B)]); lo, hi = np.nanpercentile(bs, 2.5, 0), np.nanpercentile(bs, 97.5, 0)
    return {"노출": FACTORS[a], "매개변수 집합": "+".join(meds), "총효과": e[0], "직접효과": e[1], "간접효과(차이법)": e[2], "간접 하한": lo[2], "간접 상한": hi[2], "매개 비율": e[3], "비율 하한": lo[3], "비율 상한": hi[3]}
MM = pd.DataFrame([multi_med(PRS_OF[dz], DZ) for dz in DZ] + [multi_med(PRS_OF[dz], DZ + ["sbp", "glu", "ldl", "bmi"]) for dz in DZ])
MM.to_csv(TAB / f"{PFX}table4d2_mediation_multi.csv", index=False, encoding="utf-8-sig")
print("\n다중 매개(차이법):"); print(MM.round(4).to_string(index=False))

매개분석: PRS 1 SD → 매개변수 → 해마 z (부트스트랩 1000회, 35초)
         노출 매개변수               총효과 (95% CI)               NIE (95% CI)               NDE (95% CI)    매개 비율 (95% CI)   상호작용 p  NIE와 NDE 부호 불일치
    HTN PRS  htn -0.1242 (-0.1449, -0.1049) -0.1221 (-0.1365, -0.1091) -0.0021 (-0.0228, +0.0202)   98% (84%, 118%)    0.853            False
    HTN PRS  sbp -0.0958 (-0.1163, -0.0767) -0.0096 (-0.0132, -0.0063) -0.0862 (-0.1066, -0.0674)     10% (6%, 14%)  0.00515            False
    T2D PRS   dm -0.0283 (-0.0482, -0.0089) -0.0430 (-0.0507, -0.0358) +0.0147 (-0.0061, +0.0352)  152% (86%, 436%)    0.201             True
    T2D PRS  glu -0.0278 (-0.0473, -0.0080) -0.0082 (-0.0114, -0.0047) -0.0196 (-0.0395, +0.0002)    29% (14%, 99%)    0.364            False
  LIPID PRS  lip -0.0184 (-0.0374, +0.0012) -0.0155 (-0.0237, -0.0077) -0.0029 (-0.0242, +0.0176) 84% (-207%, 653%)     0.72            False
  LIPID PRS  ldl -0.0181 (-0.0384, +0.0024) +0.0010 (+0.0000, +0.0022) -0.0192 (-0.0396, +0.0013)   

## 5. 1-표본 멘델리안 무작위화 (2-3b)

도구변수 = 형질군 PRS(대립유전자 점수), 노출 = 측정치(SBP, 공복혈당, LDL) 또는 질환 진단, 결과 = 해마 z. 2단계 최소제곱(2SLS)으로 추정하고 1단계 F 통계량, Durbin–Wu–Hausman 내생성 검정, 관찰 OLS 추정치와 비교한다. 다중 도구(형질군 lead 변이 개별 투입)로 Sargan 과대식별 검정을 하고, 고지혈증 도구에서 APOE 영역 변이를 제외한 민감도와 변이 수준 IVW, MR-Egger를 함께 제시한다. 1-표본 설계이므로 약한 도구 편향은 OLS 쪽으로 향한다는 한계를 명시한다.

In [13]:
def tsls(y, x, z, covs=COV, d=None):
    d = (A if d is None else d).dropna(subset=[y, x] + z + covs); n = len(d)
    C = np.column_stack([np.ones(n), d[covs].values]); Z = np.column_stack([C, d[z].values]); xv = d[x].values; yv = d[y].values
    # 1단계
    g = np.linalg.lstsq(Z, xv, rcond=None)[0]; xhat = Z @ g; v = xv - xhat
    g0 = np.linalg.lstsq(C, xv, rcond=None)[0]; rss0 = ((xv - C @ g0) ** 2).sum(); rss1 = (v ** 2).sum(); L = len(z)
    F = ((rss0 - rss1) / L) / (rss1 / (n - Z.shape[1])); r2_inc = (rss0 - rss1) / ((xv - xv.mean()) ** 2).sum()
    # 2단계
    Xh = np.column_stack([C, xhat]); X = np.column_stack([C, xv]); b = np.linalg.solve(Xh.T @ Xh, Xh.T @ yv); u = yv - X @ b
    s2 = (u @ u) / (n - X.shape[1]); se = np.sqrt(np.diag(s2 * np.linalg.inv(Xh.T @ Xh)))[-1]
    # OLS
    bo = np.linalg.lstsq(X, yv, rcond=None)[0]; uo = yv - X @ bo; seo = np.sqrt(np.diag((uo @ uo) / (n - X.shape[1]) * np.linalg.inv(X.T @ X)))[-1]
    # DWH
    Xd = np.column_stack([X, v]); bd = np.linalg.lstsq(Xd, yv, rcond=None)[0]; ud = yv - Xd @ bd
    sed = np.sqrt(np.diag((ud @ ud) / (n - Xd.shape[1]) * np.linalg.inv(Xd.T @ Xd)))[-1]; p_dwh = 2 * stats.t.sf(abs(bd[-1] / sed), n - Xd.shape[1])
    # Sargan
    if L > 1:
        gs = np.linalg.lstsq(Z, u, rcond=None)[0]; r2 = 1 - ((u - Z @ gs) ** 2).sum() / ((u - u.mean()) ** 2).sum(); sargan = n * r2; p_sargan = stats.chi2.sf(sargan, L - 1)
    else: sargan, p_sargan = np.nan, np.nan
    return {"n": n, "도구 수": L, "1단계 F": F, "1단계 증분 R²": r2_inc, "IV β": b[-1], "IV 하한": b[-1] - 1.96 * se, "IV 상한": b[-1] + 1.96 * se, "IV p": 2 * stats.norm.sf(abs(b[-1] / se)),
            "OLS β": bo[-1], "OLS 하한": bo[-1] - 1.96 * seo, "OLS 상한": bo[-1] + 1.96 * seo, "DWH p": p_dwh, "Sargan": sargan, "Sargan p": p_sargan}

APOE_SNPS = set(spec["annotation_groups"].get("APOE", [])) | {"rs429358", "rs7412", "rs2075650", "rs405509", "rs769446", "rs449647"}
SETS = {"HTN": [s for s in spec["prs_sets"]["HTN"] if s in LEAD.columns], "T2D": [s for s in spec["prs_sets"]["T2D"] if s in LEAD.columns], "LIPID": [s for s in spec["prs_sets"]["LIPID"] if s in LEAD.columns]}
for g, snps in SETS.items():
    for s in snps: A[f"g_{s}"] = LEAD[s].values
print("형질군 lead 집합 크기:", {g: len(v) for g, v in SETS.items()}, "| LIPID 집합 중 APOE 영역:", sorted(set(SETS["LIPID"]) & APOE_SNPS))

rows = []
for g, dz in [("HTN", "htn"), ("T2D", "dm"), ("LIPID", "lip")]:
    prs = PRS_OF[dz]; meas = MEAS[dz]
    for x, xlab in [(meas, f"{meas}(측정치, 1단위당)"), (dz, f"{dz}(진단, 0→1)")]:
        for o in OUTS:
            rows.append({"형질군": g, "노출": xlab, "결과": OUTS[o], "도구": "PRS(점수)", **tsls(o, x, [prs])})
        rows.append({"형질군": g, "노출": xlab, "결과": "해마 z", "도구": f"lead {len(SETS[g])}개 개별", **tsls("z_hip", x, [f"g_{s}" for s in SETS[g]])})
        if g == "LIPID":
            ex = [f"g_{s}" for s in SETS[g] if s not in APOE_SNPS]
            rows.append({"형질군": g, "노출": xlab, "결과": "해마 z", "도구": f"lead {len(ex)}개(APOE 제외)", **tsls("z_hip", x, ex)})
T4e = pd.DataFrame(rows); T4e.to_csv(TAB / f"{PFX}table4e_mr_2sls.csv", index=False, encoding="utf-8-sig")
T4e["IV β (95% CI)"] = [fmt(a, b, c, 4) for a, b, c in zip(T4e["IV β"], T4e["IV 하한"], T4e["IV 상한"])]
T4e["OLS β (95% CI)"] = [fmt(a, b, c, 4) for a, b, c in zip(T4e["OLS β"], T4e["OLS 하한"], T4e["OLS 상한"])]
print("1-표본 MR (2SLS), 공변량 연령, 성별, PC1–10:")
print(T4e[T4e.결과 == "해마 z"][["형질군", "노출", "도구", "1단계 F", "IV β (95% CI)", "IV p", "OLS β (95% CI)", "DWH p", "Sargan p"]].to_string(index=False, float_format=lambda v: f"{v:.3g}"))

# 변이 수준 IVW와 MR-Egger (같은 표본의 요약통계; 참고)
def ivw_egger(g, x):
    s = V[(V.kind == "lead") & (V.snp.isin(SETS[g]))]; bx, by, sy = s[f"beta_{x}"].values, s["beta_z_hip"].values, s["se_z_hip"].values
    ok = np.isfinite(bx) & np.isfinite(by) & (np.abs(bx) > 0); bx, by, sy = bx[ok], by[ok], sy[ok]
    sign = np.sign(bx); bx, by = bx * sign, by * sign                       # 노출 증가 방향으로 정렬
    w = 1 / sy ** 2; ivw = (w * bx * by).sum() / (w * bx ** 2).sum(); se_ivw = np.sqrt(1 / (w * bx ** 2).sum())
    Xe = np.column_stack([np.ones(len(bx)), bx]); We = np.diag(w); beta_e = np.linalg.solve(Xe.T @ We @ Xe, Xe.T @ We @ by)
    res = by - Xe @ beta_e; s2 = (w * res ** 2).sum() / max(len(bx) - 2, 1); cov = s2 * np.linalg.inv(Xe.T @ We @ Xe); se_e = np.sqrt(np.diag(cov))
    return {"형질군": g, "노출": x, "변이 수": int(ok.sum()), "IVW β": ivw, "IVW 하한": ivw - 1.96 * se_ivw, "IVW 상한": ivw + 1.96 * se_ivw,
            "Egger 기울기": beta_e[1], "Egger 기울기 SE": se_e[1], "Egger 절편": beta_e[0], "Egger 절편 p": 2 * stats.t.sf(abs(beta_e[0] / se_e[0]), max(len(bx) - 2, 1))}
EG = pd.DataFrame([ivw_egger(g, x) for g, x in [("HTN", "sbp"), ("HTN", "htn"), ("T2D", "glu"), ("T2D", "dm"), ("LIPID", "ldl"), ("LIPID", "lip")]])
EG.to_csv(TAB / f"{PFX}table4e2_mr_ivw_egger.csv", index=False, encoding="utf-8-sig")
print("\n변이 수준 IVW와 MR-Egger(참고, 1-표본):"); print(EG.to_string(index=False, float_format=lambda v: f"{v:.3g}"))

형질군 lead 집합 크기: {'HTN': 34, 'T2D': 36, 'LIPID': 32} | LIPID 집합 중 APOE 영역: ['rs405509', 'rs429358', 'rs449647', 'rs7412', 'rs769446']
1-표본 MR (2SLS), 공변량 연령, 성별, PC1–10:
  형질군             노출                도구    1단계 F              IV β (95% CI)     IV p             OLS β (95% CI)    DWH p  Sargan p
  HTN sbp(측정치, 1단위당)           PRS(점수)      131 -0.0543 (-0.0684, -0.0402) 4.26e-14 -0.0045 (-0.0058, -0.0033)  1.4e-18       NaN
  HTN sbp(측정치, 1단위당)       lead 34개 개별     4.78 -0.0427 (-0.0544, -0.0311) 6.54e-13 -0.0045 (-0.0058, -0.0033) 5.19e-14     0.123
  HTN   htn(진단, 0→1)           PRS(점수)  2.4e+03 -0.5566 (-0.6690, -0.4441) 3.02e-22 -0.5466 (-0.5962, -0.4971)    0.847       NaN
  HTN   htn(진단, 0→1)       lead 34개 개별       71 -0.5550 (-0.6671, -0.4429)  2.9e-22 -0.5466 (-0.5962, -0.4971)    0.871     0.249
  T2D glu(측정치, 1단위당)           PRS(점수)      258 -0.0083 (-0.0143, -0.0023)  0.00644 -0.0026 (-0.0036, -0.0016)   0.0561       NaN
  T2D glu(측정치, 1단위당)       lead 36개 개별     7.75 -0.

## 6. 보고서용 요약표(표 4)와 그림 6

In [14]:
# 표 4: 핵심 수치 모음
hip = T4a[T4a.결과 == "해마 z"].set_index("유전 요인")
def getm(prs, dz): r = T4d[(T4d.노출 == prs) & (T4d.매개변수 == dz)].iloc[0]; return r
summary = []
for g, dz, prs in [("HTN", "htn", "HTN PRS"), ("T2D", "dm", "T2D PRS"), ("LIPID", "lip", "LIPID PRS")]:
    att = T4c[(T4c.PRS == prs) & (T4c.보정 == "(b) + 질환 진단")].iloc[0]; r = getm(prs, dz); mr = T4e[(T4e.형질군 == g) & (T4e.노출.str.startswith(dz)) & (T4e.도구 == "PRS(점수)") & (T4e.결과 == "해마 z")].iloc[0]
    sc = T4b[T4b.형질군 == g].iloc[0]
    summary.append({"형질군": g, "PRS → 해마 z β/SD (95% CI)": fmt(hip.loc[prs, "β(1 SD 또는 대립유전자 1개당)"], hip.loc[prs, "CI 하한"], hip.loc[prs, "CI 상한"], 3), "Holm p": hip.loc[prs, "Holm p (21검정)"],
                    "질환 보정 후 감쇠율": f"{att.감쇠율:.0%} ({att['감쇠율 CI 하한']:.0%}, {att['감쇠율 CI 상한']:.0%})",
                    "매개 비율(진단 경유)": f"{r['매개 비율']:.0%} ({r['매개 비율 하한']:.0%}, {r['매개 비율 상한']:.0%})",
                    "MR: 진단 → 해마 z (IV β)": fmt(mr["IV β"], mr["IV 하한"], mr["IV 상한"], 3), "MR 1단계 F": round(mr["1단계 F"], 1),
                    "스캔: 질환 유의 좌위": int(sc[{"HTN": "고혈압 유의 좌위", "T2D": "당뇨 유의 좌위", "LIPID": "고지혈증 유의 좌위"}[g]]), "스캔: 해마 유의 좌위": int(sc["해마 유의 좌위"])})
T4 = pd.DataFrame(summary); T4.to_csv(TAB / f"{PFX}table4_genomic_summary.csv", index=False, encoding="utf-8-sig")
print("표 4 요약:"); print(T4.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("\nAPOE ε4 → 해마 z:", apoe["ε4 1개당 β"], "| λ_GC(해마):", round(lam["z_hip"], 3))

# 그림 6: (A) 보정 단계별 PRS 효과, (B) 매개 비율, (C) MR 대 OLS
fig, axes = plt.subplots(1, 3, figsize=(13, 4.8))
ax = axes[0]; order = ["HTN PRS", "T2D PRS", "LIPID PRS", "OBESITY PRS"]; steps = list(STEPS); cols = [COL["blue"], COL["orange"], COL["aqua"]]
for j, st in enumerate(steps):
    s = T4c[T4c.보정 == st].set_index("PRS").loc[order]; yv = np.arange(len(order)) + (j - 1) * 0.22
    ax.errorbar(s["β(1 SD당)"], yv, xerr=[s["β(1 SD당)"] - s["CI 하한"], s["CI 상한"] - s["β(1 SD당)"]], fmt="o", color=cols[j], capsize=3, ms=5, lw=1.5, label=st)
ax.axvline(0, color=COL["ink2"], lw=1); ax.set_yticks(range(len(order))); ax.set_yticklabels(order); ax.invert_yaxis(); ax.set_xlabel("해마 z 변화 (PRS 1 SD당)"); ax.set_title("A. 질환 보정 전후 PRS 효과")
ax.legend(fontsize=7.5, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=1)
ax = axes[1]; s = pd.concat([T4d[(T4d.노출 == pr) & (T4d.매개변수 == dz)] for pr, dz in [("HTN PRS", "htn"), ("T2D PRS", "dm"), ("LIPID PRS", "lip")]]).set_index("노출")
yv = np.arange(3); ax.barh(yv - 0.18, s.NIE, 0.36, color=COL["blue"], label="간접효과(질환 경유)"); ax.barh(yv + 0.18, s.NDE, 0.36, color=COL["orange"], label="직접효과")
ax.errorbar(s.NIE, yv - 0.18, xerr=[s.NIE - s["NIE 하한"], s["NIE 상한"] - s.NIE], fmt="none", ecolor=COL["ink2"], capsize=2, lw=1)
ax.errorbar(s.NDE, yv + 0.18, xerr=[s.NDE - s["NDE 하한"], s["NDE 상한"] - s.NDE], fmt="none", ecolor=COL["ink2"], capsize=2, lw=1)
xr = max(float(s["NDE 상한"].max()), float(s["NIE 상한"].max()), 0) + 0.004
for i, pm in enumerate(s["매개 비율"]): ax.text(xr, i, f"매개 비율 {pm:.0%}", va="center", ha="left", fontsize=8, color=COL["ink"])
ax.set_xlim(right=xr + 0.045)
ax.axvline(0, color=COL["ink2"], lw=1); ax.set_yticks(yv); ax.set_yticklabels(["HTN PRS → 고혈압", "T2D PRS → 당뇨", "LIPID PRS → 고지혈증"]); ax.invert_yaxis(); ax.set_xlabel("해마 z 변화 (PRS 1 SD당)"); ax.set_title("B. 자연효과 분해")
ax.legend(fontsize=7.5, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=2)
ax = axes[2]; s = T4e[(T4e.도구 == "PRS(점수)") & (T4e.결과 == "해마 z") & T4e.노출.str.contains("진단")].set_index("형질군").loc[["HTN", "T2D", "LIPID"]]
yv = np.arange(3); ax.errorbar(s["IV β"], yv - 0.15, xerr=[s["IV β"] - s["IV 하한"], s["IV 상한"] - s["IV β"]], fmt="o", color=COL["violet"], capsize=3, label="MR (2SLS, PRS 도구)")
ax.errorbar(s["OLS β"], yv + 0.15, xerr=[s["OLS β"] - s["OLS 하한"], s["OLS 상한"] - s["OLS β"]], fmt="s", color=COL["gray"], capsize=3, label="관찰 OLS")
ax.axvline(0, color=COL["ink2"], lw=1); ax.set_yticks(yv); ax.set_yticklabels(["고혈압 진단", "당뇨 진단", "고지혈증 진단"]); ax.invert_yaxis(); ax.set_xlabel("해마 z 변화 (진단 0→1)"); ax.set_title("C. 질환 → 해마: MR 대 관찰")
ax.legend(fontsize=7.5, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=2)
for a_ in axes: a_.spines[["top", "right"]].set_visible(False); a_.grid(axis="x", color="#e6e6e6", lw=0.6); a_.set_axisbelow(True)
plt.tight_layout(); plt.savefig(FIG / f"{PFX}fig6_genomic_pathway.png", dpi=150, bbox_inches="tight"); plt.close(); print("그림 저장:", f"{PFX}fig6_genomic_pathway.png")
print("\n저장 표:", ", ".join(sorted(p.name for p in TAB.glob(f"{PFX}table4*"))))

표 4 요약:
  형질군 PRS → 해마 z β/SD (95% CI)   Holm p       질환 보정 후 감쇠율      매개 비율(진단 경유)    MR: 진단 → 해마 z (IV β)  MR 1단계 F  스캔: 질환 유의 좌위  스캔: 해마 유의 좌위
  HTN  -0.095 (-0.115, -0.076) 5.42e-20   95% (77%, 124%)   98% (84%, 118%) -0.557 (-0.669, -0.444)   2.4e+03            17             1
  T2D  -0.026 (-0.046, -0.007)    0.143  152% (87%, 534%)  152% (86%, 436%) -0.309 (-0.537, -0.082)  1.27e+03            20             0
LIPID  -0.019 (-0.038, +0.001)    0.801 90% (-683%, 804%) 84% (-207%, 653%) -0.176 (-0.365, +0.012)  1.74e+03            18             0

APOE ε4 → 해마 z: +0.006 (-0.042, +0.055) | λ_GC(해마): 1.001
그림 저장: fig6_genomic_pathway.png

저장 표: table4_eval_performance.csv, table4_genomic_summary.csv, table4a2_prs_deciles.csv, table4a3_apoe.csv, table4a_prs_apoe_association.csv, table4b_scan_by_group.csv, table4c_signal_transfer.csv, table4d2_mediation_multi.csv, table4d_mediation.csv, table4e2_mr_ivw_egger.csv, table4e_mr_2sls.csv
